# 1. MetaHarness Quickstart: Memory, Policy, Execution, Evidence

This notebook builds a complete MemoRizz MetaHarness run without calling an external model. A small deterministic adapter stands in for Codex, Claude Code, or OpenHands so that every moving part is visible and reproducible.

By the end, you will be able to:

- describe the boundary between an **agent harness** and a **meta-harness**;
- create a tenant-scoped memory context pack;
- express permissions, budgets, and verification in a `HarnessTask`;
- inspect normalized events, routing evidence, and the durable run ledger; and
- discover locally installed Codex, Claude Code, and OpenHands adapters without running them.

> **Cost and safety:** every execution in this notebook is local, read-only, and free. It uses a temporary workspace that is removed in the final cell.

## The contract we are building

A vendor harness owns its internal reasoning and tool loop. MemoRizz owns the cross-harness concerns that should not change when you switch models or vendors.

```mermaid
flowchart LR
    Request[Scoped task request] --> Context[Retrieve bounded memory]
    Context --> Policy[Validate policy and budget]
    Policy --> Route[Route by capability]
    Route --> Adapter[AgentHarness adapter]
    Adapter --> Events[Normalize events and usage]
    Events --> Verify[Run host verification]
    Verify --> Result[Durable HarnessResult]
    Result --> Evidence[(Observability and learning evidence)]
```

| Layer | Owns | Does not own |
|---|---|---|
| Vendor harness | Reasoning loop, vendor tools, model interaction | Tenant memory policy or cross-vendor audit schema |
| MemoRizz MetaHarness | Memory context, routing, approval, budgets, events, verification, evidence | Reimplementation of each vendor's agent loop |
| Host application | Identity, approval decisions, workspace roots, business outcome | Model-generated claims that a risky action was approved |

## 1. Set up an isolated tutorial runtime

Use a kernel with MemoRizz installed. From a released package, install `memorizz>=0.6.0`; from the repository, use `python -m pip install -e ".[dev]"`. The code below intentionally does not modify `sys.path`, load secrets, or depend on a running database.

The tutorial creates four disposable resources:

1. a tiny Python workspace;
2. a filesystem memory provider;
3. a SQLite run ledger; and
4. a SQLite approval ledger.

In [ ]:
import shutil
import sys
import tempfile
from pathlib import Path
from pprint import pprint

import memorizz
from memorizz.approval import SQLiteApprovalStore
from memorizz.enums.memory_type import MemoryType
from memorizz.memory_provider.filesystem.provider import FileSystemConfig, FileSystemProvider
from memorizz.metaharness import (
    AdapterOutcome,
    AgentHarness,
    HarnessBudget,
    HarnessCapabilities,
    HarnessEvent,
    HarnessEventType,
    HarnessPermissions,
    HarnessTask,
    MetaHarness,
    SQLiteHarnessRunStore,
    VerificationSpec,
)

DEMO_ROOT = Path(tempfile.mkdtemp(prefix="memorizz-metaharness-quickstart-"))
WORKSPACE = DEMO_ROOT / "workspace"
WORKSPACE.mkdir()
(WORKSPACE / "app.py").write_text(
    'def release_status(checks_passed: bool) -> str:\n'
    '    return "ready" if checks_passed else "blocked"\n',
    encoding="utf-8",
)
(WORKSPACE / "verify.py").write_text(
    "import ast\nfrom pathlib import Path\nast.parse(Path('app.py').read_text(encoding='utf-8'))\n",
    encoding="utf-8",
)

provider = FileSystemProvider(
    FileSystemConfig(
        root_path=DEMO_ROOT / "memory",
        embedding_provider=None,
        lazy_vector_indexes=True,
        use_faiss=False,
    )
)

print({"memorizz_version": memorizz.__version__, "module": memorizz.__file__, "demo_root": str(DEMO_ROOT)})

## 2. Implement the smallest useful adapter

Every integration implements `AgentHarness`. `probe()` advertises capabilities without starting a run. `run()` receives the already validated workspace, the scoped memory pack, an event sink, and a cooperative cancellation signal. It returns an `AdapterOutcome`; MemoRizz turns that into a stable `HarnessResult`.

The tutorial adapter records the memory source IDs it received and emits two normalized events. A production adapter would translate its vendor's stream into the same event vocabulary.

```mermaid
sequenceDiagram
    participant M as MetaHarness
    participant A as TutorialReadHarness
    M->>A: probe()
    A-->>M: capabilities
    M->>A: run(task, workspace, context_pack, emit, cancel_event)
    A-->>M: status + message events
    A-->>M: AdapterOutcome
    M->>M: host verification + durable persistence
```

In [ ]:
class TutorialReadHarness(AgentHarness):
    name = "tutorial-read"

    def __init__(self):
        self.last_context_pack = None

    def probe(self):
        return HarnessCapabilities(
            name=self.name,
            available=True,
            version="tutorial-1",
            command="in-process educational adapter",
            structured_events=True,
            mcp=False,
            usage_reporting=True,
            metadata={"network_modes": ["none"], "task_tool_policy": True},
        )

    def run(self, task, *, workspace, context_pack, emit, cancel_event):
        self.last_context_pack = context_pack
        emit(HarnessEvent(task.run_id, HarnessEventType.STATUS, {"phase": "inspect"}))
        source_ids = list(context_pack.source_ids if context_pack else [])
        emit(
            HarnessEvent(
                task.run_id,
                HarnessEventType.MESSAGE,
                {"text": "Reviewed app.py with scoped memory", "memory_source_ids": source_ids},
            )
        )
        return AdapterOutcome(
            final_response=(
                "The release helper is deterministic and the configured release policy "
                f"was grounded in memory sources: {source_ids}."
            ),
            usage={"input_tokens": 96, "output_tokens": 24, "steps": 1},
            cost_usd=0.0,
            exit_code=0,
        )

adapter = TutorialReadHarness()
pprint(adapter.probe().to_dict())

## 3. Seed tenant-scoped memory

A harness should receive relevant context, not a dump of every stored record. MemoRizz queries supported memory types, applies `memory_id`, `user_id`, and `thread_id` isolation **before** inclusion, removes unsafe payload fields, deduplicates source IDs, and enforces a character budget.

```mermaid
flowchart LR
    Q[Task query] --> R[Retrieve by memory type]
    Scope[memory_id + user_id + thread_id] --> R
    R --> S[Sanitize and deduplicate]
    S --> B{Within context budget?}
    B -- yes --> P[Context pack with source IDs]
    B -- no --> T[Mark truncated and omit overflow]
```

The following record is visible only to the tutorial's exact memory and user scope. Its ID becomes provenance the adapter can cite.

In [ ]:
MEMORY_ID = "release-engineering"
USER_ID = "tutorial-user"
THREAD_ID = "release-42"
TASK_QUERY = "Review the release helper against our release readiness and verification policy."

policy_source_id = provider.store(
    {
        "title": "Release readiness policy",
        "content": (
            f"Applicable request: {TASK_QUERY}\n"
            "A release is ready only when source parsing succeeds and the host "
            "verification command exits with status zero."
        ),
        "user_id": USER_ID,
        "thread_id": THREAD_ID,
        "tags": ["release", "verification"],
    },
    MemoryType.KNOWLEDGE_BASE,
    memory_id=MEMORY_ID,
)
print({"policy_source_id": policy_source_id})

## 4. Create a bounded task envelope

`HarnessTask` is the portable contract shared by the SDK, CLI, UI, and MCP server. Keep policy outside the natural-language task whenever it can be represented structurally.

| Field | Purpose in this run |
|---|---|
| `harness` | Explicitly selects our educational adapter |
| memory/user/thread scope | Prevents cross-tenant and cross-thread memory leakage |
| `workspace_mode=read_only` | Prevents the adapter from being granted direct writes |
| `network=none` | Expresses that the task needs no egress |
| `mcp_access=none` | Avoids injecting the MemoRizz MCP server into this adapter |
| budget | Bounds wall time, steps, token telemetry, and event volume |
| verification | Lets the host establish success independently of the adapter's prose |

Because this is a trusted-host, read-only envelope with no network, secrets, extra tools, or governed writes, it can start without an approval proposal.

In [ ]:
service = MetaHarness(
    memory_provider=provider,
    adapters=[adapter],
    run_store=SQLiteHarnessRunStore(DEMO_ROOT / "runs.sqlite3"),
    approval_store=SQLiteApprovalStore(DEMO_ROOT / "approvals.sqlite3"),
    allowed_workspace_roots=[str(WORKSPACE)],
    context_max_chars=8_000,
)

verification_command = f'"{sys.executable}" -B verify.py'
task = HarnessTask(
    task=TASK_QUERY,
    workspace=str(WORKSPACE),
    harness="tutorial-read",
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    permissions=HarnessPermissions(
        workspace_mode="read_only",
        network="none",
        mcp_access="none",
    ),
    budget=HarnessBudget(
        max_wall_time_seconds=30,
        max_steps=4,
        max_input_tokens=500,
        max_output_tokens=200,
    ),
    verification=VerificationSpec(command=verification_command, timeout_seconds=20),
)

result = service.run(task)
pprint(result.to_dict())

## 5. Read the evidence, not only the answer

A useful agent harness result is more than text. Check these invariants:

- `status == "succeeded"` means the adapter completed and policy did not fail;
- `verified is True` means the host command independently passed;
- `ok is True` combines those conditions;
- `context_pack.source_ids` records which memory grounded the run;
- `routing.candidates` explains why an adapter was accepted or rejected; and
- events provide a normalized timeline independent of vendor log formats.

The assertions below turn the notebook into an executable specification.

In [ ]:
assert result.ok is True
assert result.status.value == "succeeded"
assert result.verified is True
assert policy_source_id in result.context_pack.source_ids
assert result.workspace_fingerprint_before == result.workspace_fingerprint_after

print("Final response:", result.final_response)
print("Context tokens (estimated):", result.context_pack.token_estimate)
print("Routing decision:")
pprint(result.routing)

In [ ]:
persisted_run = service.get_run(result.run_id)
events = service.events(result.run_id)

print("Durable status:", persisted_run["status"])
print("Normalized timeline:")
for event in events:
    print(f"  #{event['sequence']:02d} {event['type']:<13} {event['data']}")

assert any(event["type"] == "verification" for event in events)
assert events[-1]["type"] == "complete"

## 6. Discover production adapters without invoking them

`MetaHarness.from_env()` registers Codex, Claude Code, OpenHands, and—when a compatible memory provider is present—a saved-MemAgent adapter. `list_harnesses()` only probes capabilities; it does not spend tokens or start an agent.

A probe can report that an executable exists while still marking it unready because authentication, isolation, or another required policy capability is missing. Treat `ready`, not merely `available`, as the operational signal.

In [ ]:
discovery = MetaHarness.from_env(
    memory_provider=provider,
    run_store=SQLiteHarnessRunStore(DEMO_ROOT / "discovery-runs.sqlite3"),
    approval_store=SQLiteApprovalStore(DEMO_ROOT / "discovery-approvals.sqlite3"),
    allowed_workspace_roots=[str(WORKSPACE)],
)

for capability in discovery.list_harnesses():
    print(
        f"{capability['name']:<12} ready={str(capability['ready']):<5} "
        f"version={capability.get('version')!s:<12} error={capability.get('error')}"
    )

## What you should take away

The deterministic adapter was intentionally uninteresting; the surrounding contract is the lesson. The same task envelope can be routed to a different harness while scope, approvals, verification, and evidence remain stable. This separation lets MemoRizz be memory-first without pretending to be every vendor's execution loop.

Suggested exercises:

1. Store a record for a different `user_id` and prove its source ID is absent.
2. Lower `context_max_chars` and inspect `context_pack.truncated`.
3. Make the verification command fail and compare `status`, `verified`, and `ok`.
4. Add a custom event from the adapter and confirm it appears in the durable ledger.

Continue to notebook 2 for direct writes and durable human approval.

In [ ]:
discovery.close()
service.close()
provider.close()
shutil.rmtree(DEMO_ROOT, ignore_errors=True)
print("Removed tutorial resources:", DEMO_ROOT)